# Chapitre 5 - Exercices 5.11 à 5.15

Noyau de la matrice de pénalité, poids et ex-æquo, validation croisée par augmentation
de l'échantillon, contraintes des splines de plaque mince, propriétés des RKHS.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl.ch05_splines import SmoothingSpline, loocv_score
set_style()

## Exercice 5.11 : Noyau de la matrice de pénalité

**Énoncé.** Montrer que, pour une spline de lissage, le noyau de la matrice de pénalité
$\mathbf K$ (forme de Reinsch $\mathbf S_\lambda = (\mathbf I + \lambda\mathbf K)^{-1}$)
est engendré par les fonctions linéaires en $X$.

**Rappels.** Avec $x_1 < \dots < x_N$ distincts ($N \ge 2$), $\mathbf N$ la matrice
(inversible) de la base naturelle aux $x_i$ et $\mathbf K = \mathbf N^{-T}\Omega_N\mathbf
N^{-1}$ (exercice 5.9). Pour $\mathbf f \in \mathbb R^N$, soit $\theta = \mathbf
N^{-1}\mathbf f$ et $\hat f = \sum_j\theta_jN_j$ la spline naturelle interpolant les
valeurs $\mathbf f$ aux $x_i$. Alors

$$
\mathbf f^T\mathbf K\mathbf f = \theta^T\Omega_N\theta = \int\hat f''(t)^2\,dt .
$$

**Solution.** $\mathbf K$ est symétrique semi-définie positive. Pour une telle matrice,
$\mathbf K\mathbf f = 0 \iff \mathbf f^T\mathbf K\mathbf f = 0$. En effet, en diagonalisant
$\mathbf K = \sum_kd_k\mathbf u_k\mathbf u_k^T$ ($d_k \ge 0$),
$\mathbf f^T\mathbf K\mathbf f = \sum_kd_k\langle\mathbf u_k, \mathbf f\rangle^2$ est nul
ssi $\langle\mathbf u_k, \mathbf f\rangle = 0$ pour tout $k$ tel que $d_k > 0$, ssi
$\mathbf K\mathbf f = \sum_kd_k\langle\mathbf u_k, \mathbf f\rangle\mathbf u_k = 0$.

Or $\int\hat f''^2 = 0 \iff \hat f'' \equiv 0$ (fonction continue, positive, d'intégrale
nulle) $\iff \hat f$ affine $\iff f_i = a + bx_i$ pour tout $i$ (si les $f_i$ sont alignés,
la droite est une spline naturelle qui les interpole, et c'est la seule par unicité).
Donc

$$
\ker\mathbf K = \{(a + bx_i)_{i=1..N} : a, b \in \mathbb R\} = \mathrm{Vect}(\mathbf 1, \mathbf x),
$$

de dimension 2. Conséquence : $\mathbf S_\lambda\mathbf f = \mathbf f$ pour $\mathbf f$
affine, quel que soit $\lambda$ (les deux valeurs propres $\rho_k = 1$ du cours 2).

**Vérification numérique** : on reconstitue $\mathbf K = (\mathbf S_\lambda^{-1} -
\mathbf I)/\lambda$ et l'on examine ses plus petites valeurs propres et les vecteurs propres
associés.

In [2]:
rng = np.random.RandomState(0)
n = 50
x = np.sort(rng.uniform(0, 1, n))
lam = 1e-3
S = SmoothingSpline(lam=lam).fit(x, np.zeros(n)).smoother_matrix()
K = (np.linalg.inv(S) - np.eye(n)) / lam
K = (K + K.T) / 2
d, U = np.linalg.eigh(K)
print("3 plus petites valeurs propres de K :", np.round(d[:3], 6) + 0.0)
P = np.column_stack([np.ones(n), x])
proj = P @ np.linalg.lstsq(P, U[:, :2], rcond=None)[0]
print("résidu de projection des 2 premiers vecteurs propres sur Vect(1, x) :", np.abs(proj - U[:, :2]).max())
v = 2 - 3 * x
print("||K v|| / (||K|| ||v||) pour v = 2 - 3x :", np.linalg.norm(K @ v) / (np.linalg.norm(K, 2) * np.linalg.norm(v)))

3 plus petites valeurs propres de K : [0.       0.       9.301486]
résidu de projection des 2 premiers vecteurs propres sur Vect(1, x) : 3.015522886951061e-09
||K v|| / (||K|| ||v||) pour v = 2 - 3x : 7.367887060510247e-15


> **À retenir.** $\mathbf f^T\mathbf K\mathbf f$ est la courbure de l'interpolant
> naturel ; elle s'annule exactement pour les vecteurs alignés : le noyau de $\mathbf K$
> est $\mathrm{Vect}(\mathbf 1, \mathbf x)$, et ces composantes ne sont jamais rétrécies.